# Kaggle — CellViT nucleus feature extraction

Run the preflight first. Full extraction starts only after the exact checkpoint, postprocessor, DINO forward, paths, GPU, and writable disk have passed. Internet must be enabled unless `DINO_MODEL` points to a mounted local model directory.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/CryAndRRich/codapath.git'
REPO_BRANCH = 'tiendung'
CODAPATH = Path('/kaggle/working/codapath')
if (CODAPATH / '.git').is_dir():
    subprocess.check_call(['git', '-C', str(CODAPATH), 'fetch', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', str(CODAPATH), 'switch', REPO_BRANCH])
    subprocess.check_call(['git', '-C', str(CODAPATH), 'pull', '--ff-only', 'origin', REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f'{CODAPATH} exists but is not a Git repository')
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(CODAPATH)])
os.chdir(CODAPATH)
actual_branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print('repo:', CODAPATH, '| branch:', actual_branch)

In [ ]:
# Do not install CellViT's full dependency tree over Kaggle's PyTorch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--require-hashes', '-r', 'requirements-kaggle-cellvit.txt'])
import importlib.util
needed = {'transformers': 'transformers>=4.27', 'yaml': 'PyYAML>=6.0', 'einops': 'einops>=0.6.1'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

# These scientific packages are normally preinstalled on Kaggle. Preflight
# will report the exact missing/ABI-broken import instead of silently changing them.
print('dependency setup complete')

In [ ]:
# ---- EDIT ONLY THIS CELL ----
DATASET = 'pathmnist'  # pathmnist | skintissue; HistoSet needs per-source MPP
SEED = 42
DATA_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/cryandrrich/nckh2026'),
    Path('/kaggle/input/nckh2026'),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])
DATA_PATH = str(DATA_ROOT / 'pathmnist_224.npz')
CHECKPOINT_CANDIDATES = [
    DATA_ROOT / 'CellViT-256-x40-AMP.pth',
    Path('/kaggle/input/cellvit-checkpoints/CellViT-256-x40-AMP.pth'),
]
CHECKPOINT_PATH = str(next((p for p in CHECKPOINT_CANDIDATES if p.is_file()), CHECKPOINT_CANDIDATES[0]))
# PathMNIST source pixels are 0.5 MPP. Match MODEL_MPP/MAGNIFICATION to checkpoint.
INPUT_MPP = 0.5
MODEL_MPP = 0.25
MAGNIFICATION = 40
CACHE_DIR = '/kaggle/working/nucleus_features'
DINO_MODEL = 'facebook/dinov2-base'  # or a mounted local model directory
BATCH_SIZE = 2
DINO_CROP_BATCH_SIZE = 32
SMOKE_SAMPLES = 8  # spread across the train set for runtime/cache estimate
MAX_ESTIMATED_HOURS = 10.0  # fail before wasting a Kaggle session
MAX_CELLS_PER_PATCH = None  # if pilot is too slow, use an explicit ablation cap
OVERWRITE = False

assert Path(DATA_PATH).exists(), DATA_PATH
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
assert MAGNIFICATION in (20, 40)
assert MODEL_MPP == {20: 0.5, 40: 0.25}[MAGNIFICATION]
assert not str(CACHE_DIR).startswith('/kaggle/input/'), 'Kaggle input is read-only'

In [ ]:
# Exact one-batch integration test. Do not continue if this cell fails.
preflight = [
    sys.executable, 'scripts/preflight_nucleus_kaggle.py',
    '--dataset', DATASET, '--data_path', DATA_PATH,
    '--checkpoint', CHECKPOINT_PATH, '--cache_dir', CACHE_DIR,
    '--input_mpp', str(INPUT_MPP), '--model_mpp', str(MODEL_MPP),
    '--magnification', str(MAGNIFICATION),
    '--vit_name', DINO_MODEL, '--seed', str(SEED),
    '--smoke_samples', str(SMOKE_SAMPLES),
    '--cellvit_batch_size', str(BATCH_SIZE),
    '--dino_crop_batch_size', str(DINO_CROP_BATCH_SIZE),
    '--max_estimated_hours', str(MAX_ESTIMATED_HOURS),
]
if MAX_CELLS_PER_PATCH is not None:
    preflight += ['--max_cells_per_patch', str(MAX_CELLS_PER_PATCH)]
subprocess.check_call(preflight)

In [ ]:
completed_manifest = Path(CACHE_DIR) / f'{DATASET}_seed{SEED}' / 'manifest.json'
command = [
    sys.executable, 'scripts/extract_nucleus_features.py',
    '--dataset', DATASET, '--data_path', DATA_PATH,
    '--checkpoint', CHECKPOINT_PATH, '--cache_dir', CACHE_DIR,
    '--input_mpp', str(INPUT_MPP), '--model_mpp', str(MODEL_MPP),
    '--magnification', str(MAGNIFICATION),
    '--vit_name', DINO_MODEL, '--seed', str(SEED), '--device', 'cuda',
    '--batch_size', str(BATCH_SIZE),
    '--dino_crop_batch_size', str(DINO_CROP_BATCH_SIZE),
]
if MAX_CELLS_PER_PATCH is not None:
    command += ['--max_cells_per_patch', str(MAX_CELLS_PER_PATCH)]
if OVERWRITE:
    command.append('--overwrite')
if completed_manifest.is_file() and not OVERWRITE:
    print('completed cache already exists; skipping extraction:', completed_manifest)
else:
    subprocess.check_call(command)

In [ ]:
# Validate the finished artifact before saving it as a Kaggle output dataset.
import json
from load_data import get_data_loaders, get_sample_ids
from nucleus.cache import load_nucleus_cache
cache_path = Path(CACHE_DIR) / f'{DATASET}_seed{SEED}'
manifest = json.loads((cache_path / 'manifest.json').read_text())
required = ['offsets.npy', 'confidence.npy', 'sample_ids.npy',
            'cellvit_embeddings.npy', 'cell_dino_features.npy', 'manifest.json']
missing = [name for name in required if not (cache_path / name).exists()]
assert not missing, missing
train_loader, _, _ = get_data_loaders(DATA_PATH, SEED, verbose=True)
cache = load_nucleus_cache(cache_path, expected_sample_ids=get_sample_ids(train_loader.dataset))
assert cache.num_patches == len(train_loader.dataset)
print(json.dumps(manifest, indent=2))
print('cache size GiB:', sum(p.stat().st_size for p in cache_path.rglob('*') if p.is_file()) / 2**30)

In [ ]:
# Visual QC: red boundaries must follow nuclei, not background/whole glands.
from PIL import Image
from IPython.display import display
for path in sorted((cache_path / 'qc').glob('*.png'))[:8]:
    display(Image.open(path))